In [ ]:
import sys
import os

# Configure path for imports
current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

print("FULL PREPROCESSING + GEOCODING")
print("=" * 80)

try:
    print("Checking dependencies...")
    
    # Check basic dependencies
    try:
        import requests
        import tqdm
        print("Basic dependencies available")
    except ImportError as e:
        print(f"Missing dependencies: {e}")
        print("Install with: pip install requests tqdm")
        sys.exit(1)
    
    # Check geospatial dependencies (optional)
    try:
        import geopandas
        import shapely
        print("Geospatial dependencies available")
        geo_available = True
    except ImportError:
        print("Geospatial dependencies not available")
        print("Geocoding will work but without district validation")
        print("To install: pip install geopandas")
        geo_available = False
    
    print("\nChecking data...")
    from src.preprocessing.data_loader import get_available_csv_files
    
    files_available = get_available_csv_files()
    missing_files = files_available['missing_persons']
    found_files = files_available['found_persons']
    
    if not missing_files:
        print("No missing persons files found")
        print("Expected location: data/raw/missing_persons/")
        print("Run scraper first to generate data")
        sys.exit(1)
    
    print("Data found:")
    print(f"   Missing: {len(missing_files)} files")
    print(f"   Found: {len(found_files)} files")
    
    # List missing files
    for file in missing_files:
        print(f"      {file}")
    
    print("\nVerifying shapefiles...")
    from src.preprocessing.geocoder import get_peru_shapefile_path
    
    shapefile_path = get_peru_shapefile_path()
    if shapefile_path:
        print(f"Shapefile found: {shapefile_path}")
        if geo_available:
            print("   District validation enabled")
        else:
            print("   District validation disabled (geopandas missing)")
    else:
        print("Peru shapefile not found")
        print("Expected location: data/external/peru_shapes/")
        print("Geocoding will work without district validation")
    
    print("\nIMPORTANT INFORMATION:")
    print("   This process may take 2-4 HOURS")
    print("   Will make thousands of external API calls (Photon)")
    print("   Requires a stable internet connection")
    print("   Will process thousands of records")
    
    # Estimate records safely (handling encoding)
    try:
        total_records = 0
        for file in missing_files:
            file_path = f'data/raw/missing_persons/{file}'
            try:
                # Try UTF-8 first
                with open(file_path, 'r', encoding='utf-8') as f:
                    line_count = len(f.readlines())
            except UnicodeDecodeError:
                try:
                    # If UTF-8 fails, try UTF-8-sig (BOM)
                    with open(file_path, 'r', encoding='utf-8-sig') as f:
                        line_count = len(f.readlines())
                except UnicodeDecodeError:
                    try:
                        # If that fails, try latin-1
                        with open(file_path, 'r', encoding='latin-1') as f:
                            line_count = len(f.readlines())
                    except:
                        line_count = 10000  # Default estimate
            total_records += line_count
        
        estimated_hours = (total_records / 10000) * 0.5  # Approximation
        print(f"   Estimated records: {total_records:,}")
        print(f"   Estimated time: {estimated_hours:.1f} hours")
        
    except Exception as e:
        print(f"   Could not estimate exact time: {e}")
        print("   Estimated time: 1-3 hours")
    
    confirm = input("\nContinue with full processing? (y/N): ").lower()
    if confirm != 'y':
        print("Processing cancelled by user")
        sys.exit(0)
    
    print("\nSTARTING FULL PROCESSING...")
    print("=" * 80)
    
    # Run full preprocessing
    from src.preprocessing.main_preprocessing import main_preprocessing_workflow
    
    final_dataset, summary = main_preprocessing_workflow()
    
    print("\nPROCESSING COMPLETED SUCCESSFULLY!")
    print("=" * 80)
    
    # Display final statistics
    print("FINAL STATISTICS:")
    print(f"   Total records processed: {len(final_dataset):,}")
    print(f"   Columns in final dataset: {len(final_dataset.columns)}")
    
    # Target variable
    if 'appeared_status' in final_dataset.columns:
        appeared = final_dataset['appeared_status'].sum()
        total = len(final_dataset)
        rate = (appeared / total) * 100
        print(f"   Persons appeared: {appeared:,} ({rate:.2f}%)")
    
    # Geocoding
    if 'Latitud' in final_dataset.columns:
        geocoded = final_dataset['Latitud'].notna().sum()
        rate = (geocoded / len(final_dataset)) * 100
        print(f"   Geocoded addresses: {geocoded:,} ({rate:.2f}%)")
    
    if 'En_Distrito' in final_dataset.columns:
        valid_districts = final_dataset['En_Distrito'].sum()
        print(f"   In valid districts: {valid_districts:,}")
    
    # Time calculations
    if 'hours_to_appear' in final_dataset.columns:
        valid_hours_appear = final_dataset['hours_to_appear'].notna().sum()
        print(f"   Hours to appear calculated: {valid_hours_appear:,}")
    
    if 'hours_to_report' in final_dataset.columns:
        valid_hours_report = final_dataset['hours_to_report'].notna().sum()
        print(f"   Hours to report calculated: {valid_hours_report:,}")
    
    # Generated files
    print("\nGENERATED FILES:")
    import glob
    processed_files = glob.glob("data/processed/*.csv")
    for file in processed_files:
        size = os.path.getsize(file) / 1024 / 1024  # MB
        print(f"   {os.path.basename(file)} ({size:.1f} MB)")
    
    # Sample of final data
    print("\nSAMPLE OF FINAL DATA:")
    sample_cols = ['full_name', 'age_cleaned', 'appeared_status', 'Latitud', 'Longitud']
    available_cols = [col for col in sample_cols if col in final_dataset.columns]
    
    if available_cols:
        sample = final_dataset[available_cols].head(3)
        
        for _, row in sample.iterrows():
            name = str(row.get('full_name', 'N/A'))[:30]
            age = row.get('age_cleaned', 'N/A')
            appeared = row.get('appeared_status', 'N/A')
            lat = row.get('Latitud', 'N/A')
            lon = row.get('Longitud', 'N/A')
            
            # Format coordinates if available
            if lat != 'N/A' and lon != 'N/A' and lat is not None and lon is not None:
                coords = f"({lat:.4f}, {lon:.4f})"
            else:
                coords = "(no coordinates)"
            
            print(f"   {name}: {age} years, appeared: {appeared}, coords: {coords}")
    
    print("\nALL SUCCESS!")
    print("Final dataset ready for analysis")
    print("Main file: data/processed/datos_combinados_geocodificados.csv")
    print("The message 'No geocodified data file found' will no longer appear")
    
except ImportError as e:
    print(f"\nIMPORT ERROR: {e}")
    print("Check that the files are in src/preprocessing/")
    
except Exception as e:
    print(f"\nERROR DURING EXECUTION: {e}")
    import traceback
    traceback.print_exc()
    
    print("\nPOSSIBLE SOLUTIONS:")
    print("1. Check internet connection")
    print("2. Check that scraper data is in data/raw/")
    print("3. If the error is encoding-related, the files may have special characters")
    print("4. Try again (external APIs sometimes fail)")

print("\nPROCESSING COMPLETED")
print("=" * 80)
